<a href="https://colab.research.google.com/github/Ragib301/Sentiment-Analysis-BERT-on-Yelp-Reviews/blob/main/Sentiment_Analysis_with_BERT_Neural_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Install and Import Dependencies

In [ ]:
!pip install -q torch torchaudio torchvision torchtext torchdata
!pip install -q transformers requests beautifulsoup4 pandas numpy deep-translator
print(f"Setup Completed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 840.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.1 MB/s eta 0:00:00
Setup Completed!


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import requests
from bs4 import BeautifulSoup
from deep_translator import GoogleTranslator
import re

# 2. Instantiate Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')
model = AutoModelForSequenceClassification.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

# 3. Encode & Calculate Sentiment

In [ ]:
tokens = tokenizer.encode('Ahh, It was good but could have done better!', return_tensors='pt')

In [ ]:
result = model(tokens)
result

SequenceClassifierOutput(loss=None, logits=tensor([[-1.3351,  0.6217,  2.0665,  0.6396, -1.7678]],
       grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [ ]:
rating = int(torch.argmax(result.logits))+1
rating

3

# 4. Collect Reviews

In [ ]:
regex = re.compile(r'^\s+|\s+$')

def clean_text(text):
  if re.search(r'[\u0980-\u09FF]+', text):
    return GoogleTranslator(source='bn', target='en').translate(text)
  else:
    return re.sub(regex, '', text)

In [ ]:
r = requests.get(
    "https://store.roboticsbd.com/development-boards/94-8-arduino-uno-r3-robotics-bangladesh.html")
soup = BeautifulSoup(r.text, 'html.parser')
content = soup.find_all("div", class_="col-md-12 content-block")
reviews = [clean_text(result.text) for result in content]
print(reviews)

['quality was very nice', 'Good', 'good', 'Good', 'Worked', 'Excellent', '.', 'Very good product', 'The product is original and works fine and is in pristine condition', 'good', 'Good product.', 'Good', 'I did not get the USB programming cable which was supposed to be included in the parcel', 'Good', 'good quality', 'Good quality', "The product works fine just as intended.Though the back side could look better but it\\'s ignorable.", 'Good', 'excellent product', 'Nice', 'Good', 'Good', 'Good', 'Good product', 'Products is good.', 'I am so happy to buy the product from Robotics BD.com because they have been able to deliver the product in a very quick time and their product is quite low than other companies.', 'good', 'Good Component', 'I brought it from their office . every product was intact , fresh , similar to the picture given on the RoboticsBD website. I have not used it yet not but I will update here.', 'Product is very good quality', 'Good Product', 'Good', 'Little bit of overpri

In [ ]:
reviews[8]

'The product is original and works fine and is in pristine condition'

# 5. Load Reviews into DataFrame and Score

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.DataFrame(np.array(reviews), columns=['Reviews'])
df.head()

,Reviews
0,quality was very nice
1,Good
2,good
3,Good
4,Worked


In [ ]:
def sentiment_score(review):
  tokens = tokenizer.encode(review, return_tensors='pt')
  result = model(tokens)
  return int(torch.argmax(result.logits))+1

In [ ]:
num = 30
print(df['Reviews'].iloc[num])
sentiment_score(df['Reviews'].iloc[num])

Good Product


4

In [ ]:
df['Sentiment'] = df['Reviews'].apply(lambda x: sentiment_score(x[:512]))
df

,Reviews,Sentiment
0,quality was very nice,5
1,Good,4
2,good,4
3,Good,4
4,Worked,4
...,...,...
351,yeah its great ! your service is good because...,4
352,alert('test'),4
353,A very good one. Price is fine. East to instal...,4
354,So far so good!,5
